# Chapter 8 Exercise — Coffee Maker Constraint Checking

Apply `verify_satisfaction()` and stale record detection to the coffee maker model.

## Problem

Your model from Chapter 7 has `BrewUnit` variants with different throughput values. Work through these steps:

1. Add `assert satisfy heating by weak;` alongside the existing satisfaction    declarations in your Chapter 7 model (or use the worked-example model from    `tests/fixtures/probe.sysml` as a reference). Call `verify_satisfaction()` and    confirm that `weak` holds=False for the `HeatingReq` requirement    (weak.power=400, threshold 600 W).

2. Create an `asserted_solution` ReviewRecord for the `weak` violation: set    `engineering_conclusion='refuted'`, add a non-empty `counterevidence` field,    and assert `validate_record(record) == []`.

3. Change the HeatingReq threshold from 600 W to 350 W in the model source string.    Confirm that `check_stale()` returns True for the record created in step 2.

Verify: `verify_satisfaction()` shows `weak` holds=False; `validate_record` returns `[]`; `check_stale` fires after the threshold change.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your Chapter 7 coffee maker model here (or use the worked-example model).
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")
if not model.ok:
    print(format_diagnostics(model.diagnostics))


In [ ]:
# Step 1: verify_satisfaction
verdicts = model.verify_satisfaction()
print(f"Verdicts: {len(verdicts)}")
for v in verdicts:
    status = "PASS" if v.holds else "FAIL"
    print(f"  [{status}] {v.element}")

# Find the weak variant's verdict for HeatingReq
weak_verdict = next(
    (v for v in verdicts if "weak" in (v.element or "").lower() and not v.holds),
    None,
)
if weak_verdict:
    print(f"\nViolation witness: {weak_verdict.element!r} holds={weak_verdict.holds}")
else:
    print("\nweak violation not found — check the assert satisfy declaration")


In [ ]:
# Step 2: violation witness ReviewRecord
from toaster.evidence import ReviewRecord, hash_content, validate_record

if weak_verdict:
    record = ReviewRecord(
        identifier="AS-C08-EX",
        kind="asserted_solution",
        claim="# Your claim about the weak variant failing HeatingReq",
        model_ref="CoffeeDemo::weak",
        content_hash=hash_content(source),
        scope="CoffeeDemo",
        criteria="verify_satisfaction() returns holds=False for weak",
        rationale="# Why the weak variant fails HeatingReq",
        counterevidence="# What this violation does and does not establish",
        residual_uncertainties="# What is not captured by this check",
        disposition="pending",
        dependency_freshness="current",
        engineering_conclusion="refuted",
        record_kind="worked_example",
    )
    errors = validate_record(record)
    print(f"Validation errors: {errors}")


In [ ]:
# Step 3: stale detection after threshold change
from toaster.evidence import check_stale

if weak_verdict:
    assert not check_stale(record, source), "Record should be current"
    print(f"Before change: check_stale={check_stale(record, source)}")

    # Change the HeatingReq threshold
    revised = source.replace("heater.power >= 600.0", "heater.power >= 350.0")
    revised_model = conn.load_from_content(revised, strict=False)
    assert revised_model.ok

    stale = check_stale(record, revised)
    print(f"After threshold change: check_stale={stale}")
    assert stale, "Record should be stale after threshold change"
    print("Record requires re-review.")
conn.close()
